# Exploratory study: top 100 UK groups by origin

This is the separate **popularity-first** companion to the city-first scene
depth study. It asks where the most-listened-to UK musical groups in a frozen
candidate universe originated.

It does **not** rank city scene depth. Starting with popularity structurally
favours places that produced globally dominant acts, so these results describe
geographic concentration within this selected top 100 only.

In [1]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "reference/uk_fua_top20_2024.csv").exists()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate the uk-music-cities repository root")

SNAPSHOT_ID = "20260718T204522Z"
BANDS_PATH = ROOT / "data/processed/popularity_first_top100_20260718T204522Z_bands.csv"
ORIGINS_PATH = ROOT / "data/processed/popularity_first_top100_20260718T204522Z_origins.csv"
AUDIT_PATH = ROOT / "data/interim/popularity_first_top100_20260718T204522Z_identity_audit.csv"
REPORT_PATH = ROOT / "data/processed/popularity_first_top100_20260718T204522Z_report.json"
ARTIFACT_DIR = ROOT / "artifacts/top100_popularity_first/20260718T204522Z"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

bands = pd.read_csv(BANDS_PATH, keep_default_na=False)
origins = pd.read_csv(ORIGINS_PATH, keep_default_na=False)
audit = pd.read_csv(AUDIT_PATH, keep_default_na=False)
report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))

assert len(bands) == 100
assert bands["popularity_rank"].tolist() == list(range(1, 101))
assert bands["returned_spotify_id"].nunique() == 100
assert origins["band_count"].sum() == 100
assert report["radiohead_selected"]

captured_at = bands["stats_extracted_at_utc"].iloc[0]
display(Markdown(
    f"**Frozen reach snapshot:** `{SNAPSHOT_ID}` ({captured_at}) · "
    f"**{report['candidate_ids']:,} candidate Spotify IDs** · "
    "**100 selected groups**"
))

**Frozen reach snapshot:** `20260718T204522Z` (2026-07-18T20:45:22.974241+00:00) · **1,775 candidate Spotify IDs** · **100 selected groups**

## 01. Selection rule and lineage

1. Start with the archived Wikidata query response for entities returned as UK
   musical groups, bands, or duos with a Spotify artist ID.
2. Capture current Spotify monthly listeners for those IDs.
3. Accept exact display-name matches plus explicitly reviewed aliases; reject
   unresolved name mismatches.
4. Collapse Spotify redirects to one canonical artist page and exclude
   orchestras from the "bands/groups" selection.
5. Select the 100 largest monthly-listener counts, then map their reported
   formation places to conservative origin clusters.

The source snapshot, every response, identity audit, and manual override are
retained. This notebook performs no network calls.

In [2]:
lineage = pd.DataFrame(
    [
        {
            "stage": "Candidate universe",
            "frozen input": "data/raw/wikidata/uk_group_candidates_with_spotify_20260718T201100Z.json",
            "definition": "Archived Wikidata UK-group query response",
        },
        {
            "stage": "Reach capture",
            "frozen input": report["inputs"]["metrics"],
            "definition": "Spotify monthly listeners captured at one UTC time",
        },
        {
            "stage": "Identity review",
            "frozen input": str(AUDIT_PATH.relative_to(ROOT)),
            "definition": "Exact names, reviewed aliases, redirects, exclusions",
        },
        {
            "stage": "Origin review",
            "frozen input": "reference/popularity_first_overrides_20260718.csv",
            "definition": "Formation-place overrides with source URLs",
        },
    ]
)
display(lineage.style.hide(axis="index"))

stage,frozen input,definition
Candidate universe,data/raw/wikidata/uk_group_candidates_with_spotify_20260718T201100Z.json,Archived Wikidata UK-group query response
Reach capture,data/processed/uk_group_spotify_metrics_20260718T204522Z.csv,Spotify monthly listeners captured at one UTC time
Identity review,data/interim/popularity_first_top100_20260718T204522Z_identity_audit.csv,"Exact names, reviewed aliases, redirects, exclusions"
Origin review,reference/popularity_first_overrides_20260718.csv,Formation-place overrides with source URLs


## 02. Capture and identity quality

The counts below keep the incomplete and rejected rows visible. A missing
listener metric or mismatched name is not silently converted to zero.

In [3]:
qa = pd.DataFrame(
    [
        ("Candidate Spotify IDs", report["candidate_ids"]),
        ("Pages with listener metrics", report["metrics_rows"]),
        ("Pages without a listener metric", report["metric_failures"]),
        ("Display-name mismatches sent to review", report["identity_name_reviews"]),
        ("Accepted identities after review", report["identity_accepted_rows"]),
        ("Orchestra rows excluded from selection pool", report["orchestra_rows_excluded"]),
        ("Redirect-duplicate rows removed", report["redirect_duplicate_rows"]),
        ("Selected groups", report["selected_bands"]),
        ("Selected groups with resolved origins", report["origin_resolved_bands"]),
    ],
    columns=["check", "count"],
)
display(qa.style.hide(axis="index"))

reviewed_exceptions = bands.loc[
    bands["identity_status"].eq("accepted_reviewed_alias")
    | bands["origin_resolution"].eq("reviewed_override"),
    [
        "popularity_rank",
        "band_name",
        "spotify_name",
        "identity_status",
        "formation_label",
        "origin_cluster",
        "reason",
        "source_url",
    ],
]
display(Markdown("**Reviewed exceptions that enter the selected 100**"))
display(reviewed_exceptions.style.hide(axis="index"))

check,count
Candidate Spotify IDs,1775
Pages with listener metrics,1749
Pages without a listener metric,26
Display-name mismatches sent to review,121
Accepted identities after review,1695
Orchestra rows excluded from selection pool,27
Redirect-duplicate rows removed,7
Selected groups,100
Selected groups with resolved origins,100


**Reviewed exceptions that enter the selected 100**

popularity_rank,band_name,spotify_name,identity_status,formation_label,origin_cluster,reason,source_url
13,Bee Gees,Bee Gees,accepted_exact_name,Los Hornos,Redcliffe,"The archived Wikidata formation label Los Hornos is erroneous; the Bee Gees' official history places the group's naming and first record deal in Redcliffe, Queensland.",https://www.beegees.com/bee-gees-way-redcliffe-australia/
30,The Outfield,The Outfield,accepted_exact_name,Manchester,London,The band's official biography describes it as London-based and rooted in London's East End; Manchester is unsupported.,https://theoutfield.com/bio
35,Eurythmics,Eurythmics,accepted_exact_name,Sunderland,London,The duo formed in London; Sunderland is Dave Stewart's earlier home scene.,https://rockhall.com/wp-content/uploads/2024/03/Eurythmics_RNRHF_Final_2022-LR.pdf
36,Little Mix,Little Mix,accepted_exact_name,England,London,Wikidata records only England; Apple Music records London as the formation place,https://music.apple.com/gb/artist/little-mix/477515548
79,The Wanted,The Wanted,accepted_exact_name,England,London,Wikidata records only England; Apple Music records London as the formation place,https://music.apple.com/us/artist/the-wanted/3445978
90,The Pretenders,Pretenders,accepted_reviewed_alias,Hereford,Hereford,Spotify differs only by the leading article: The Pretenders / Pretenders,https://open.spotify.com/artist/0GByy3DcfbQwDvXGCWmzv9
96,Katrina and the Waves,Katrina & The Waves,accepted_reviewed_alias,Cambridge,Cambridge,Spotify uses an ampersand where Wikidata spells out and,https://open.spotify.com/artist/2TzHIUhVpeeDxyJPpQfnV3
97,LF System,LF SYSTEM,accepted_exact_name,,West Lothian,Wikidata formation place is missing; the artist's official biography describes both members as West Lothian natives,https://lfsystemmusic.com/about


## 03. The selected top 100

Monthly listeners are a volatile global reach measure, not a timeless quality
score. The full table is shown so the cutoff and every origin assignment remain
auditable.

In [4]:
top100_table = bands[
    [
        "popularity_rank",
        "spotify_name",
        "monthly_listeners",
        "formation_label",
        "origin_cluster",
        "origin_resolution",
    ]
].rename(
    columns={
        "popularity_rank": "rank",
        "spotify_name": "group",
        "monthly_listeners": "monthly listeners",
        "formation_label": "reported formation place",
        "origin_cluster": "origin cluster",
        "origin_resolution": "origin rule",
    }
)
display(
    top100_table.style
    .hide(axis="index")
    .format({"monthly listeners": "{:,.0f}"})
)

rank,group,monthly listeners,reported formation place,origin cluster,origin rule
1,Coldplay,"92,034,104",London,London,reported_place
2,Arctic Monkeys,"52,301,870",Sheffield,Sheffield,reported_place
3,Queen,"51,601,234",London,London,reported_place
4,Fleetwood Mac,"50,959,935",London,London,reported_place
5,One Direction,"45,724,043",London,London,reported_place
6,Radiohead,"43,168,156",Abingdon-on-Thames,Oxford,editorial_city_cluster
7,The Police,"39,887,109",London,London,reported_place
8,Oasis,"39,676,473",Manchester,Manchester,reported_place
9,Gorillaz,"38,173,160",London,London,reported_place
10,The Beatles,"37,927,636",Liverpool,Liverpool,reported_place


In [5]:
plot_top = bands.head(20).sort_values("monthly_listeners")
fig, ax = plt.subplots(figsize=(10, 7))
colors = ["#1f77b4" if origin == "London" else "#b8bec6" for origin in plot_top["origin_cluster"]]
ax.barh(plot_top["spotify_name"], plot_top["monthly_listeners"] / 1_000_000, color=colors)
ax.set(
    title="Top 20 selected UK groups by captured monthly listeners",
    xlabel="Monthly listeners (millions)",
    ylabel="",
)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
path = ARTIFACT_DIR / "01_top20_groups.png"
fig.savefig(path, dpi=180, bbox_inches="tight")
plt.show()
display(Markdown("*Blue denotes a London origin cluster; grey denotes every other origin.*"))

/var/folders/jk/2s13yhh11zx3vw8rzxvwhcxm0000gn/T/ipykernel_27093/408689534.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


*Blue denotes a London origin cluster; grey denotes every other origin.*

## 04. Geographic concentration

Counts answer "how many of the selected 100 came from each origin?" Listener
share answers "how much of the selected sample's captured reach came from each
origin?" Both use the same frozen top 100.

In [6]:
origin_top = origins.head(12).sort_values("band_count")
fig, ax = plt.subplots(figsize=(9, 6))
colors = ["#d95f02" if origin == "London" else "#b8bec6" for origin in origin_top["origin_cluster"]]
ax.barh(origin_top["origin_cluster"], origin_top["band_count"], color=colors)
ax.set(
    title="Most frequent origin clusters in the selected top 100",
    xlabel="Number of selected groups",
    ylabel="",
)
ax.spines[["top", "right"]].set_visible(False)
for y, value in enumerate(origin_top["band_count"]):
    ax.text(value + 0.35, y, f"{int(value)}", va="center")
fig.tight_layout()
path = ARTIFACT_DIR / "02_origin_band_counts.png"
fig.savefig(path, dpi=180, bbox_inches="tight")
plt.show()

display(
    origins.head(15).style
    .hide(axis="index")
    .format(
        {
            "monthly_listeners_total": "{:,.0f}",
            "band_share": "{:.1%}",
            "listener_share": "{:.1%}",
        }
    )
)

/var/folders/jk/2s13yhh11zx3vw8rzxvwhcxm0000gn/T/ipykernel_27093/931626229.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


origin_cluster,band_count,monthly_listeners_total,band_share,listener_share
London,46,"835,709,009",46.0%,51.1%
Manchester,10,"147,968,643",10.0%,9.0%
Birmingham,5,"69,081,043",5.0%,4.2%
Sheffield,4,"82,887,435",4.0%,5.1%
Liverpool,3,"50,053,015",3.0%,3.1%
Oxford,2,"70,261,364",2.0%,4.3%
Cambridge,2,"41,124,400",2.0%,2.5%
Glasgow,2,"20,436,452",2.0%,1.2%
Bristol,2,"14,651,842",2.0%,0.9%
Leeds,2,"13,877,184",2.0%,0.8%


In [7]:
london = origins.loc[origins["origin_cluster"].eq("London")].iloc[0]
manchester = origins.loc[origins["origin_cluster"].eq("Manchester")].iloc[0]
radiohead = bands.loc[bands["spotify_name"].eq("Radiohead")].iloc[0]
effective_origins_count = 1 / report["origin_hhi_band_count_resolved"]
effective_origins_reach = 1 / report["origin_hhi_reach_resolved"]

display(Markdown(
    f"""## 05. Result

- **London contributes {int(london['band_count'])} of the selected 100 groups**
  ({london['listener_share']:.1%} of captured reach).
- **Manchester contributes {int(manchester['band_count'])} groups**
  ({manchester['listener_share']:.1%} of reach).
- **Radiohead ranks #{int(radiohead['popularity_rank'])}** and maps from
  Abingdon-on-Thames to the **Oxford** origin cluster—the kind of small-place,
  giant-band case the city-first design intentionally cannot discover.
- The resolved-origin HHI is
  **{report['origin_hhi_band_count_resolved']:.3f} by group count** and
  **{report['origin_hhi_reach_resolved']:.3f} by listener reach**. Expressed as
  inverse-HHI “effective origins,” that is about
  **{effective_origins_count:.1f}** and **{effective_origins_reach:.1f}**
  equally sized origins respectively.

This is strong concentration within the popularity-selected sample. It is not
evidence that London has the deepest population-normalized scene; that is the
different question answered by the city-first study."""
))

## 05. Result

- **London contributes 46 of the selected 100 groups**
  (51.1% of captured reach).
- **Manchester contributes 10 groups**
  (9.0% of reach).
- **Radiohead ranks #6** and maps from
  Abingdon-on-Thames to the **Oxford** origin cluster—the kind of small-place,
  giant-band case the city-first design intentionally cannot discover.
- The resolved-origin HHI is
  **0.231 by group count** and
  **0.279 by listener reach**. Expressed as
  inverse-HHI “effective origins,” that is about
  **4.3** and **3.6**
  equally sized origins respectively.

This is strong concentration within the popularity-selected sample. It is not
evidence that London has the deepest population-normalized scene; that is the
different question answered by the city-first study.

## 06. Limitations

- The candidate frame inherits Wikidata coverage, classifications, Spotify-ID
  errors, and multi-country edge cases. It is reproducible, but not exhaustive.
- "UK group" means the entity was returned by the archived UK-country query.
  Cases such as the Bee Gees or America demonstrate why nationality and
  formation place are not always equivalent.
- Monthly listeners change daily and measure global reach. Results belong to
  this snapshot only.
- Spotify's web-player artist overview supplied the metric. The endpoint is
  undocumented; raw responses are retained for audit.
- Origin is usually the Wikidata formation place. A small reviewed override
  table fills missing or overly broad values, and conservative editorial
  clustering joins named districts to London, Manchester, or Oxford.
- HHI depends on how origins are clustered and on the top-100 cutoff.

The conclusion is narrow: **among this frozen popularity-selected
top 100, origins and listener reach are geographically concentrated, especially
in London.**